# 06 -- Demo Cases (Nepal Flood / Assam Flood / Ahmedabad Aircraft Crash)

Runs the full pipeline (severity -> priority -> recommendation) end-to-end
on the 3 required demo scenarios, using the JSON files in
`backend/data/demo_cases/`.

In [1]:
import sys, json
sys.path.append("../backend")

from services.disaster_service import load_demo_case, get_disaster_factors
from recommendation.scoring import calculate_priority
from recommendation.mitigation import get_immediate_safety, get_temporary_mitigation

case_names = ["nepal_flood", "assam_flood", "ahmedabad_crash"]
cases = {name: load_demo_case(name) for name in case_names}

for name, case in cases.items():
    print(f"{name}: {len(case['sites'])} sites, disaster_type={case['disaster_type']}")
    print("  Relevant extra factors for this disaster type:", get_disaster_factors(case['disaster_type']))

nepal_flood: 2 sites, disaster_type=flood
  Relevant extra factors for this disaster type: ['flood_depth', 'road_blockage', 'bridge_access', 'rainfall_intensity']
assam_flood: 2 sites, disaster_type=flood
  Relevant extra factors for this disaster type: ['flood_depth', 'road_blockage', 'bridge_access', 'rainfall_intensity']
ahmedabad_crash: 2 sites, disaster_type=aircraft_crash
  Relevant extra factors for this disaster type: ['fire', 'smoke', 'debris_field', 'secondary_explosion_risk']


## Score every site in every case

In [2]:
import pandas as pd

rows = []
for case_name, case in cases.items():
    for site in case["sites"]:
        result = calculate_priority(site)
        rows.append({
            "case": case_name,
            "site_id": site["site_id"],
            "damage_type": site.get("damage_type"),
            "priority_score": result["priority_score"],
            "priority_level": result["priority_level"],
        })

df = pd.DataFrame(rows).sort_values(["case", "priority_score"], ascending=[True, False])
df

,case,site_id,damage_type,priority_score,priority_level
4,ahmedabad_crash,AH_BUILDING_01,fire,61.1,HIGH
5,ahmedabad_crash,AH_ROAD_01,debris,49.1,MEDIUM
2,assam_flood,AS_ROAD_01,flooding,64.3,HIGH
3,assam_flood,AS_SCHOOL_01,flooding,42.6,MEDIUM
0,nepal_flood,NP_BRIDGE_01,flooding,58.9,HIGH
1,nepal_flood,NP_ROAD_02,debris,47.6,MEDIUM


## Generate recommendations for the highest-priority site in each case

In [3]:
for case_name, case in cases.items():
    top_site = max(case["sites"], key=lambda s: calculate_priority(s)["priority_score"])
    damage_type = top_site.get("damage_type", "structural_damage")
    severity = top_site.get("damage_severity", 5)
    print(f"\n=== {case_name} -- top priority site: {top_site['site_id']} ===")
    print("Immediate safety:", get_immediate_safety(damage_type, severity))
    print("Temporary mitigation:", get_temporary_mitigation(damage_type))


=== nepal_flood -- top priority site: NP_BRIDGE_01 ===
Immediate safety: Avoid route. Do not attempt to cross. Warn public via signage.
Temporary mitigation: ['Close road/route', 'Signpost alternate route', 'Temporary drainage if feasible']

=== assam_flood -- top priority site: AS_ROAD_01 ===
Immediate safety: Avoid route. Do not attempt to cross. Warn public via signage.
Temporary mitigation: ['Close road/route', 'Signpost alternate route', 'Temporary drainage if feasible']

=== ahmedabad_crash -- top priority site: AH_BUILDING_01 ===
Immediate safety: URGENT: Evacuate the area. Alert fire services. Restrict public access.
Temporary mitigation: ['Establish safety perimeter', 'Coordinate with fire services', 'Assess secondary fire risk']


## Next steps

- Replace the demo JSON site data with real detection output once your
  team uploads actual Nepal/Assam/Ahmedabad imagery through `/api/upload`.
- These 3 cases are exactly what the frontend dashboard and PDF report
  are designed to display -- run the FastAPI backend and visit
  `frontend/dashboard.html` to see them rendered.